# Week 3, day 2 — Worksheet 06 SOLUTIONS: duplicates   (L05)

Executed in the lab image (pandas 3.0.5) against the real
`data/customers_messy.csv`. Every quoted number is what it actually printed.

Questions 5 and 7 are the ones to re-read. One row that looks identical to
another is not a duplicate, and `unique()` does not return what the deck says.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 06 — Duplicates. Run this once.
import pandas as pd

cust = pd.read_csv("data/customers_messy.csv")

print("shape:", cust.shape)
print(cust.head())

PART A — finding them

### Question 1

`440` rows, **`29`** flagged as duplicates.

`duplicated()` returns a boolean Series, one entry per row, so summing it
counts the repeats. The flagged rows start at index 400 — the tail of the
file, which is where a careless append usually puts them.

In [ ]:
flags = cust.duplicated()
print("rows:", len(cust))
print("flagged as duplicate:", flags.sum())
print()
print(cust[flags].head(6).to_string())

### Question 2

`keep="first"` -> `29`. `keep=False` -> **`58`**. Difference `29`.

Two different questions.

The default answers **'which rows should I delete'** — it marks every
occurrence after the first, so deleting them leaves exactly one of each.

`keep=False` answers **'which rows are involved in a duplication'** — it
flags the original too. That is what you want for *inspecting* the problem,
because the original and its copy sit next to each other and you can see
whether they really are the same record.

Use `keep=False` to look, the default to delete.

In [ ]:
first = cust.duplicated().sum()
allof = cust.duplicated(keep=False).sum()
print("keep='first' (default):", first)
print("keep=False:            ", allof)
print("difference:            ", allof - first)
print()
print("-> keep=False also flags the ORIGINAL of each repeated pair.")

### Question 3

Every row for that `CustomerID` is byte-identical — `drop_duplicates()` on the group leaves one row.

Confirmed before deleting anything, which is the deck's point. 'Ana, 25'
appearing twice could be one record entered twice or two different people;
only the data and the domain can tell you which.

Here the rows are genuinely identical in every column, so keeping one loses
nothing.

In [ ]:
dupe_id = cust[cust.duplicated()]["CustomerID"].iloc[0]
group = cust[cust["CustomerID"] == dupe_id]
print("CustomerID:", dupe_id)
print(group.to_string())
print()
print("rows:", len(group))
print("all identical:", group.drop_duplicates().shape[0] == 1)

### Question 4

`440` -> **`411`** rows, `29` removed. -> `cust` itself is unchanged.

The count matches Q1 exactly, which is the check.

And `drop_duplicates` returns a copy, like `drop` and `sort_values` before
it. This is the third method in the course with that behaviour; assume it
for every Pandas method until you have checked otherwise.

In [ ]:
clean = cust.drop_duplicates()
print("before:", len(cust))
print("after: ", len(clean))
print("removed:", len(cust) - len(clean))
print()
print("original still:", len(cust), "-> drop_duplicates returns a copy")

PART B — the ones that only look identical

### Question 5

`411` rows but only **`400` distinct `CustomerID`s** -> **11** IDs still appear twice. -> e.g. `'Bill Donatelli'` and `'Bill Donatelli '`.

Eleven customers survived de-duplication twice over, and the only
difference between their two rows is a **trailing space**.

`drop_duplicates()` compares whole rows for exact equality. `'Bill
Donatelli'` and `'Bill Donatelli '` are different strings, so the pair is
not a duplicate by that definition — and the printed output gives you no
hint, because a trailing space is invisible.

This is the same `repr()` lesson as the alignment sheet in the 30/08 class.
When two things that should match do not, `repr()` them.

It is also the most common near-duplicate in real data: the same record
re-exported by a system that pads its fields.

In [ ]:
clean = cust.drop_duplicates()
print("rows after drop_duplicates:", len(clean))
print("distinct CustomerIDs:      ", clean["CustomerID"].nunique())
print()
counts = clean["CustomerID"].value_counts()
repeated = counts[counts > 1]
print("IDs still appearing more than once:", len(repeated))
if len(repeated):
    cid = repeated.index[0]
    rows = clean[clean["CustomerID"] == cid]
    print()
    for _, r in rows.iterrows():
        print("  id=%s name=%r province=%r" % (r["CustomerID"], r["CustomerName"], r["Province"]))

### Question 6

**41** of 440 names carry stray whitespace. -> `drop_duplicates()` gives `411` before stripping and **`400`** after.

Stripping first removes another 11 rows, landing on the 400 distinct
customers Q5 said were there.

The order matters and it is the wrong way round by default. De-duplicating
before normalising leaves near-duplicates behind; normalising first catches
them. **Clean the values, then remove the duplicates.**

`.str.strip()` on every text column immediately after loading costs one
line and removes an entire category of this problem.

In [ ]:
padded = (cust["CustomerName"] != cust["CustomerName"].str.strip()).sum()
print("names with stray whitespace:", padded, "of", len(cust))
print()
fixed = cust.copy()
fixed["CustomerName"] = fixed["CustomerName"].str.strip()
print("drop_duplicates before stripping:", len(cust.drop_duplicates()))
print("drop_duplicates after stripping: ", len(fixed.drop_duplicates()))

### Question 7

`unique()` -> an **`ArrowStringArray`**, `dtype: str`. -> **not** a `numpy.ndarray`.

The deck makes two claims on that slide — 'returns distinct values as a
NumPy array' and `dtype=object` — and this Pandas contradicts both.

Both were true for years. Pandas 3 made a real string dtype the default and
backs it with Arrow, so a text column's `unique()` gives an Arrow-backed
array of `str`.

It still behaves like a sequence — iterate, index, `len()`, `sorted()` — so
most code is unaffected. Code that is affected is code that checked
`isinstance(x, np.ndarray)` or `dtype == object`, and it fails silently by
taking the wrong branch rather than raising.

In [ ]:
u = cust["Region"].unique()
print(repr(u))
print()
print("type: ", type(u).__name__)
print("dtype:", u.dtype)
print("is it a numpy.ndarray?", isinstance(u, __import__("numpy").ndarray))
print()
print("deck says: numpy array, dtype=object")

PART C — duplicates on a subset

### Question 8

`subset=["CustomerID"]` -> `400` rows either way, but `keep="first"` and `keep="last"` are **not the same rows**. -> full-row de-duplication gave `411`.

Both give 400 because there are 400 distinct IDs — the count is decided by
the subset, not by `keep`. Which *row* survives for each ID is decided by
`keep`, and Q9 shows the two disagreeing.

Using `subset` is an assertion: you are saying `CustomerID` identifies a
customer and any disagreement in the other columns does not matter. That is
often right. It is worth writing down that you have decided it, because the
code reads identically whether the assertion holds or not.

In [ ]:
by_id_first = cust.drop_duplicates(subset=["CustomerID"], keep="first")
by_id_last = cust.drop_duplicates(subset=["CustomerID"], keep="last")
print("keep='first':", len(by_id_first))
print("keep='last': ", len(by_id_last))
print("same rows:", by_id_first.equals(by_id_last))
print()
print("full-row drop_duplicates gave:", len(cust.drop_duplicates()))

### Question 9

For `68464052`, `keep="first"` keeps `'Bill Donatelli'` and `keep="last"` keeps `'Bill Donatelli '`.

One argument decides whether the clean spelling or the padded one survives
into everything downstream — every join, every `groupby`, every report.

And the default is `keep="first"`, which here means 'whichever copy the
file happened to list first'. That is row order, which is not a property of
the customer.

When duplicate rows genuinely differ, `keep` is not a tie-break, it is a
data-quality decision made by position. If one version is better than the
other — more recent, more complete, better formatted — sort deliberately
before de-duplicating so `keep="first"` means something.

In [ ]:
counts = cust["CustomerID"].value_counts()
multi = counts[counts > 1].index
target = None
for cid in multi:
    rows = cust[cust["CustomerID"] == cid]
    if len(rows.drop_duplicates()) > 1:
        target = cid
        break

print("CustomerID with differing rows:", target)
if target is not None:
    rows = cust[cust["CustomerID"] == target]
    print("all rows for that id:")
    for _, r in rows.iterrows():
        print("   name=%r province=%r" % (r["CustomerName"], r["Province"]))
    print()
    f = cust.drop_duplicates(subset=["CustomerID"], keep="first")
    l = cust.drop_duplicates(subset=["CustomerID"], keep="last")
    print("kept by first:", repr(f[f["CustomerID"] == target]["CustomerName"].iloc[0]))
    print("kept by last: ", repr(l[l["CustomerID"] == target]["CustomerName"].iloc[0]))

### Question 10

`subset=["Customer_ID"]` -> **raises** `KeyError: Index(['Customer_ID'], dtype='str')`.

One underscore. The column is `CustomerID`.

This is the well-behaved end of the sheet: `subset` validates against the
columns and stops you. Compare it with `drop(columns=[...],
errors="ignore")` from the 30/08 class, where the same typo silently did
nothing and left the data unchanged — and with Q5, where a one-character
difference in the *data* produced no error at all and left 11 duplicate
customers in a file that looked clean.

Three ways to be wrong by one character, and only one of them tells you.

In [ ]:
print("columns:", list(cust.columns))
print(cust.drop_duplicates(subset=["Customer_ID"]))